In [2]:
!pip install --upgrade transformers

In [3]:
!pip install transformers datasets torch scikit-learn

In [4]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_csv("/Users/bimsaraserasinghe/Downloads/DSGP/Pre-Processed Csvs/balanced_emotion_dataset (3).csv")

df = df[["Emotion", "Text"]]   # Make sure columns are correct
df.dropna(inplace=True)

df.head()

,Emotion,Text
0,Disguist,இதற்கு காரணம் தமிழக அரசில் கட்சியின்ர் மற்றும்...
1,Sadness,சார் நீங்கள் மது கடையை திறக்ககூடாது சார் நீங்க...
2,Ambiguous,யாரு சொன்னது தாடி வைக்கிறது இஸ்லாமிய வழக்கம்.
3,Fear,அதை சேய வேண்டாம் ஆபத்து
4,Love,🙏🙏🙏 ஓம் சாய் ராம் 🙏🙏🙏🌻🌻🌻🌻🌹🌹🌹🌹🌺🌺🌺🌺❤️❤️❤️❤️❤️❤️❤️


In [6]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Emotion"])

print(label_encoder.classes_)

['Ambiguous' 'Anger' 'Anticipation' 'Disguist' 'Fear' 'Joy' 'Love'
 'Neutral' 'Sadness' 'Surprise' 'Trust']


In [7]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["Text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [8]:
train_dataset = Dataset.from_dict({
    "text": list(train_texts),
    "label": list(train_labels)
})

val_dataset = Dataset.from_dict({
    "text": list(val_texts),
    "label": list(val_labels)
})

In [9]:
model_name = "jusgowiturs/autotrain-tamil_emotion_11_tamilbert-2710380899"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3405.50it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
XLMRobertaForSequenceClassification LOAD REPORT from: jusgowiturs/autotrain-tamil_emotion_11_tamilbert-2710380899
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 10651/10651 [00:00<00:00, 39966.05 examples/s]


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # use this instead if needed
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.093868,0.913149,0.692799,0.676811,0.682690,0.692799
2,0.658033,0.511906,0.841142,0.832254,0.834898,0.841142
3,0.410588,0.333362,0.900573,0.895419,0.896326,0.900573
4,0.244850,0.333500,0.916158,0.908616,0.914661,0.916158
5,0.170253,0.269536,0.941132,0.938262,0.939150,0.941132
6,0.129502,0.278769,0.945451,0.942946,0.943946,0.945451


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.41s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.57s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.19s/it]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shard

TrainOutput(global_step=15978, training_loss=0.5277496497373525, metrics={'train_runtime': 31350.0081, 'train_samples_per_second': 8.153, 'train_steps_per_second': 0.51, 'total_flos': 1.68141550353408e+16, 'train_loss': 0.5277496497373525, 'epoch': 6.0})

In [15]:
trainer.save_model("Tamil_emotion_model")
tokenizer.save_pretrained("Tamil_emotion_model")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]


('Tamil_emotion_model/tokenizer_config.json',
 'Tamil_emotion_model/tokenizer.json')

In [17]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="fine_tuned_emotion_model",
    tokenizer="fine_tuned_emotion_model"
)

def emotion_chatbot():
    print("🤖 Fine-Tuned Emotion Chatbot")
    print("Type 'exit' to stop\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: Take care ❤️")
            break

        result = emotion_classifier(user_input)[0]
        print(f"Bot: Emotion → {result['label']}")
        print(f"Confidence → {result['score']:.2f}\n")



OSError: fine_tuned_emotion_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
emotion_chatbot()

NameError: name 'emotion_chatbot' is not defined